# Qwen3.5-4B SFT Training with Unsloth (10k Samples & Token Length Filter)

This notebook contains the complete interactive pipeline to fine-tune the Qwen 3.5 4B model using Unsloth on the `upwitu/trash_draft_am` dataset. It includes robust data loading, conversation length analysis, token filtering, and final weight merging. All cells are wrapped with try-except diagnostic blocks for easy troubleshooting.

In [ ]:
import os
import sys
import torch
import torch.utils

try:
    # 1. SYSTEM COMPATIBILITY PATCHES (Monkey patching older PyTorch versions)
    # Must run BEFORE importing unsloth/transformers/trl to avoid torch.int1 and register_constant errors
    print("Applying system compatibility patches...")
    for i in range(1, 8):
        int_attr = f"int{i}"
        uint_attr = f"uint{i}"
        if not hasattr(torch, int_attr):
            setattr(torch, int_attr, torch.int8)
        if not hasattr(torch, uint_attr):
            setattr(torch, uint_attr, torch.uint8)

    if not hasattr(torch.utils, "_pytree"):
        import torch.utils._pytree
    if not hasattr(torch.utils._pytree, "register_constant"):
        torch.utils._pytree.register_constant = lambda cls: cls

    # 2. IMPORT UNSLOTH FIRST (Strictly before trl/transformers/peft for maximum speed & stability)
    import unsloth

    # 3. PREVENT TRANSFORMERS FROM LOADING TORCHAO (Avoid AttributeError issues)
    import transformers
    transformers.utils.import_utils.is_torchao_available = lambda *args, **kwargs: False
    transformers.utils.is_torchao_available = lambda *args, **kwargs: False
    if hasattr(transformers.utils.import_utils, "_torchao_available"):
        transformers.utils.import_utils._torchao_available = False

    # Unset DDP environment variables for safe single GPU training
    for key in ["WORLD_SIZE", "RANK", "LOCAL_RANK", "MASTER_ADDR", "MASTER_PORT"]:
        if key in os.environ:
            del os.environ[key]

    print("✅ System compatibility patches applied & Unsloth imported first!")
except Exception as e:
    print("❌ Failure in system compatibility patches or Unsloth import:", str(e))
    raise e

In [ ]:
try:
    import gc
    def clear_gpu_memory():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            print("Released GPU cache memory.")

    clear_gpu_memory()
except Exception as e:
    print("❌ Failure in clearing GPU memory:", str(e))
    raise e

In [ ]:
try:
    # Load libraries
    from datasets import Dataset
    from trl import SFTTrainer, SFTConfig
    from unsloth import FastLanguageModel, train_on_responses_only
    from huggingface_hub import hf_hub_download
    import pandas as pd
    import numpy as np
    import json

    MODEL_NAME = "Qwen/Qwen3.5-4B"
    PERSISTENT_DIR = "./outputs"
    ADAPTER_PATH = os.path.join(PERSISTENT_DIR, "sft_lora_adapter")
    MERGED_PATH = os.path.join(PERSISTENT_DIR, "sft_merged_model")

    os.makedirs(PERSISTENT_DIR, exist_ok=True)
    print("Libraries loaded successfully.")
except Exception as e:
    print("❌ Failure in importing libraries:", str(e))
    raise e

## Load Raw Dataset in Pandas

In [ ]:
try:
    print("Downloading dataset raw jsonl files directly from HF Hub...")
    jsonl_files = [
        "data/interactive_agent_disambiguation.jsonl",
        "data/interactive_agent_hallucination.jsonl",
        "data/search_disambiguation.jsonl",
        "data/search_hallucination.jsonl"
    ]

    dfs = []
    for filename in jsonl_files:
        print(f"Downloading {filename}...")
        local_file_path = hf_hub_download(
            repo_id="upwitu/trash_draft_am",
            filename=filename,
            repo_type="dataset"
        )
        dfs.append(pd.read_json(local_file_path, lines=True))

    print("Concatenating splits into a single DataFrame...")
    full_df = pd.concat(dfs, ignore_index=True)
    print(f"Combined dataset size: {len(full_df)} rows")
except Exception as e:
    print("❌ Failure in downloading or concatenating dataset:", str(e))
    raise e

## Initialize Tokenizer & Model (Unsloth)

In [ ]:
try:
    print("Loading model and processor via Unsloth...")
    model, processor = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=4096,
        dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        load_in_4bit=False,   # Keep full precision since we have A100
        trust_remote_code=True
)

    # Extract the actual tokenizer from the processor
    tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor

    print("Configuring LoRA PEFT...")
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
    print("LoRA configured successfully.")
    model.print_trainable_parameters()
except Exception as e:
    print("❌ Failure in loading model or configuring LoRA:", str(e))
    raise e

## Format Templates & Compute Lengths in Pandas

In [ ]:
def clean_messages(messages, row_idx=0):
    import json
    if not isinstance(messages, list):
        return []
    
    cleaned = []
    for msg_idx, msg in enumerate(messages):
        if not isinstance(msg, dict):
            continue
        msg = msg.copy()
        
        # Normalize tool_calls format if present
        if "tool_calls" in msg:
            tc = msg["tool_calls"]
            # If tool_calls was stored as a JSON string instead of parsed list
            if isinstance(tc, str):
                try:
                    tc = json.loads(tc)
                except Exception as e_tc:
                    print(f"[Row {row_idx}] Failed to decode tool_calls JSON string:", e_tc)
                    tc = None
            
            if isinstance(tc, list):
                cleaned_tc = []
                for tc_idx, call in enumerate(tc):
                    if isinstance(call, dict):
                        call = call.copy()
                        
                        # Normalize non-standard `{"name": "...", "arguments": "..."}` to standard format
                        if "name" in call and "arguments" in call and "function" not in call:
                            name = call["name"]
                            args = call["arguments"]
                            # Decode string to dict if needed for Qwen template compatibility
                            if isinstance(args, str):
                                try:
                                    args = json.loads(args)
                                except Exception as e_arg1:
                                    print(f"[Row {row_idx}] Failed to decode non-standard arguments:", e_arg1)
                            call = {
                                "id": f"call_{row_idx}_{msg_idx}_{tc_idx}",
                                "type": "function",
                                "function": {
                                    "name": name,
                                    "arguments": args
                                }
                            }
                        elif "function" in call and isinstance(call["function"], dict):
                            func = call["function"].copy()
                            # Decode string to dict if needed for Qwen template compatibility
                            if "arguments" in func and isinstance(func["arguments"], str):
                                try:
                                    func["arguments"] = json.loads(func["arguments"])
                                except Exception as e_arg2:
                                    print(f"[Row {row_idx}] Failed to decode standard arguments string:", e_arg2)
                            call["function"] = func
                            
                            # Ensure id and type exist in standard format
                            if "id" not in call:
                                call["id"] = f"call_{row_idx}_{msg_idx}_{tc_idx}"
                            if "type" not in call:
                                call["type"] = "function"
                                
                        cleaned_tc.append(call)
                msg["tool_calls"] = cleaned_tc
            else:
                msg.pop("tool_calls", None)
                
        cleaned.append(msg)
    return cleaned

def compute_formatted_text_and_length(row):
    # Standardize types and extract fields
    tools = row.get("tools")
    # If tools is empty or not a list, set to None to avoid Jinja filter mapping errors
    if not isinstance(tools, list) or len(tools) == 0:
        tools = None
        
    messages = row.get("augmented_messages")
    # Clean and validate augmented messages structure
    messages = clean_messages(messages, row_idx=row.name)
    
    # Safeguard: if there are no messages, return empty text to avoid IndexError
    if len(messages) == 0:
        return pd.Series({"text": "", "length": 0})
        
    # Apply chat template with error catching
    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            tools=tools
        )
        # Fast token count estimation via tokenizer encode
        tokens = tokenizer.encode(text)
        return pd.Series({"text": text, "length": len(tokens)})
    except Exception as e:
        print(f"\n❌ Formatting failed at dataframe row index: {row.name}")
        print("Tools:", tools)
        print("Messages:", messages)
        raise e

try:
    print("Applying template and computing token lengths directly in Pandas (takes about 1-2 minutes)... ")
    length_df = full_df.apply(compute_formatted_text_and_length, axis=1)
    
    # IMPORTANT: Drop existing 'text' and 'length' columns if they already exist from a previous run to avoid duplicates
    full_df = full_df.drop(columns=["text", "length"], errors="ignore")
    full_df = pd.concat([full_df, length_df], axis=1)

    # Compute statistics for non-empty lengths
    valid_lengths = full_df[full_df["length"] > 0]["length"].tolist()
    print("\n--- Token Length Statistics ---")
    print(f"Minimum length: {np.min(valid_lengths)} tokens")
    print(f"Maximum length: {np.max(valid_lengths)} tokens")
    print(f"Mean length:    {np.mean(valid_lengths):.1f} tokens")
    print(f"Median length:  {np.median(valid_lengths)} tokens")
except Exception as e:
    print("❌ Failure in applying chat templates and analyzing lengths:", str(e))
    raise e

## Filter Data (< 3000 tokens), Limit Samples (10k), and Convert to Arrow Dataset

**NOTE**: Since the system prompt and tools list for this dataset are extremely detailed (occupying >2200 tokens alone), the minimum conversation length is 2244 tokens. We set the length filter limit to **3000 tokens** to capture the shortest 25% of the dataset (~7700 samples), which prevents the dataset from being empty and ensures fast, memory-efficient training.

In [ ]:
try:
    # 1. Filter out conversations with length >= 3000 tokens and exclude empty conversations (length == 0)
    # Note: 3000 is a safe threshold since system prompt + tools takes ~2200+ tokens.
    print("Filtering out conversations containing >= 3000 tokens or empty conversations...")
    filtered_df = full_df[(full_df["length"] < 3000) & (full_df["length"] > 0)]
    print(f"Total samples under 3000 tokens: {len(filtered_df)} (out of {len(full_df)})")
    
    # SAFEGUARD: Raise error early if dataset is empty
    if len(filtered_df) == 0:
        raise ValueError("No samples found under 3000 tokens. Please make sure to re-run the previous cell (Cell 6) to recompute lengths.")

    # 2. Limit to 10,000 samples for training
    max_train_samples = 10000
    train_size = min(max_train_samples, len(filtered_df))
    print(f"Selecting first {train_size} filtered samples for training...")
    limited_df = filtered_df.head(train_size)

    # 3. Keep ONLY clean columns (text and length) to completely bypass PyArrow schema validation errors with tools column
    clean_df = limited_df[["text", "length"]].copy()

    # 4. Convert clean dataframe to Hugging Face Dataset
    print("Converting clean pandas DataFrame to Hugging Face Dataset...")
    dataset = Dataset.from_pandas(clean_df, preserve_index=False)
    print(f"Dataset successfully loaded and parsed! Dataset size: {len(dataset)}")

    # 5. Train/Validation Split (90% train, 10% test)
    dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
    train_dataset = dataset_split["train"]
    eval_dataset = dataset_split["test"]
    print(f"Final Split: {len(train_dataset)} train samples, {len(eval_dataset)} validation samples")
except Exception as e:
    print("❌ Failure in filtering, limiting, or dataset splitting:", str(e))
    raise e

## SFT Trainer & Loss Masking Configuration

In [ ]:
try:
    sft_config = SFTConfig(
        dataset_text_field="text",
        max_seq_length=4096,
        output_dir=PERSISTENT_DIR,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        gradient_accumulation_steps=4,
        num_train_epochs=3.0,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        fp16=False,
        bf16=True,
        optim="paged_adamw_8bit",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=5,
        report_to="none"
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer
)

    # Apply loss masking: train only on assistant's output + thoughts
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n<think>\n",
    )
    print("Trainer initialized and response loss masking applied.")
except Exception as e:
    print("❌ Failure in SFT Config or Trainer initialization:", str(e))
    raise e

## Run Training

In [ ]:
try:
    # Double check if train_dataset has samples
    if len(train_dataset) == 0:
        raise ValueError("train_dataset is empty! Run previous cells starting from Cell 6.")
    print("Starting training loop...")
    trainer.train()
    print("Training finished successfully!")
except Exception as e:
    print("❌ Failure during SFT training loop:", str(e))
    raise e

## Plot SFT Training Loss

In [ ]:
try:
    %matplotlib inline
    import matplotlib.pyplot as plt

    if hasattr(trainer, "state") and trainer.state.log_history:
        history = trainer.state.log_history
        train_steps = []
        train_losses = []
        eval_steps = []
        eval_losses = []
        
        for log in history:
            step = log.get("step")
            if "loss" in log:
                train_steps.append(step)
                train_losses.append(log["loss"])
            if "eval_loss" in log:
                eval_steps.append(step)
                eval_losses.append(log["eval_loss"])
                
        plt.figure(figsize=(10, 5))
        if train_losses:
            plt.plot(train_steps, train_losses, label="Train Loss", color="red", marker="o")
        if eval_losses:
            plt.plot(eval_steps, eval_losses, label="Validation Loss", color="blue", marker="s")
        plt.xlabel("Step")
        plt.ylabel("Loss")
        plt.title("SFT Loss Curve Comparison")
        plt.legend()
        plt.grid(True)
        plt.show()
    else:
        print("No log history found.")
except Exception as e:
    print("⚠️ Failed to plot loss curve (but SFT was successful!):", str(e))

## Save PEFT Adapter and Merge Model

In [ ]:
try:
    print(f"Saving adapter to: {ADAPTER_PATH}")
    model.save_pretrained(ADAPTER_PATH)
    tokenizer.save_pretrained(ADAPTER_PATH)

    print("Merging and saving model weights (16-bit)...")
    model.save_pretrained_merged(MERGED_PATH, tokenizer, save_method="merged_16bit")
    print(f"Merged model saved successfully to: {MERGED_PATH}")
except Exception as e:
    print("❌ Failure in saving adapter or merging weights:", str(e))
    raise e